In [ ]:
# Example 1: FileSystemEventHandler — define a custom handler and inspect its dispatch table.
from watchdog.events import FileSystemEventHandler, FileSystemEvent

class MyHandler(FileSystemEventHandler):
    def __init__(self):
        super().__init__()
        self.seen = []

    def on_any_event(self, event: FileSystemEvent) -> None:
        self.seen.append((event.event_type, event.src_path))

handler = MyHandler()
print(f"Handler class: {type(handler).__name__}")
print(f"Has on_any_event: {callable(getattr(handler, 'on_any_event', None))}")
print(f"Has on_created: {callable(getattr(handler, 'on_created', None))}")
print(f"Has on_modified: {callable(getattr(handler, 'on_modified', None))}")
print(f"Has on_deleted: {callable(getattr(handler, 'on_deleted', None))}")
print(f"Has on_moved: {callable(getattr(handler, 'on_moved', None))}")
assert isinstance(handler, FileSystemEventHandler)

In [ ]:
# Example 2: PatternMatchingEventHandler — pattern matching filter for events.
from watchdog.events import PatternMatchingEventHandler

handler = PatternMatchingEventHandler(
    patterns=["*.txt", "*.log"],
    ignore_patterns=["*.tmp"],
    ignore_directories=True,
    case_sensitive=False,
)

print(f"patterns = {handler.patterns}")
print(f"ignore_patterns = {handler.ignore_patterns}")
print(f"ignore_directories = {handler.ignore_directories}")
print(f"case_sensitive = {handler.case_sensitive}")

assert handler.patterns == ["*.txt", "*.log"]
assert handler.ignore_patterns == ["*.tmp"]
assert handler.ignore_directories is True

In [ ]:
# Example 3: Observer — start, schedule, and stop with a real temp dir + filesystem event.
import os
import tempfile
import time

from watchdog.events import FileSystemEventHandler
from watchdog.observers import Observer

events_seen = []

class RecordingHandler(FileSystemEventHandler):
    def on_any_event(self, event):
        events_seen.append((event.event_type, os.path.basename(event.src_path)))

with tempfile.TemporaryDirectory() as tmpdir:
    observer = Observer()
    observer.schedule(RecordingHandler(), tmpdir, recursive=False)
    observer.start()
    try:
        with open(os.path.join(tmpdir, "hello.txt"), "w") as f:
            f.write("hi")
        time.sleep(1.0)
    finally:
        observer.stop()
        observer.join(timeout=5)

print(f"observer was alive, captured {len(events_seen)} event(s)")
assert observer.is_alive() is False

In [ ]:
# Example 4: LoggingEventHandler + event dispatch directly (no real fs needed).
import logging

from watchdog.events import (
    LoggingEventHandler,
    FileCreatedEvent,
    FileModifiedEvent,
    FileDeletedEvent,
    DirCreatedEvent,
)

logging.basicConfig(level=logging.INFO, force=True)
handler = LoggingEventHandler()

events = [
    FileCreatedEvent("/tmp/example.txt"),
    FileModifiedEvent("/tmp/example.txt"),
    FileDeletedEvent("/tmp/example.txt"),
    DirCreatedEvent("/tmp/example_dir"),
]

for ev in events:
    handler.dispatch(ev)
    print(f"dispatched {type(ev).__name__}: src={ev.src_path}, type={ev.event_type}")

assert events[0].event_type == "created"
assert events[2].event_type == "deleted"

In [ ]:
# Example 5: PollingObserver — pure-python observer that doesn't depend on inotify/FSEvents.
import os
import tempfile
import time

from watchdog.events import FileSystemEventHandler
from watchdog.observers.polling import PollingObserver

captured = []

class H(FileSystemEventHandler):
    def on_created(self, event):
        captured.append(("created", os.path.basename(event.src_path)))
    def on_modified(self, event):
        captured.append(("modified", os.path.basename(event.src_path)))

with tempfile.TemporaryDirectory() as tmpdir:
    obs = PollingObserver(timeout=0.2)
    obs.schedule(H(), tmpdir, recursive=False)
    obs.start()
    try:
        path = os.path.join(tmpdir, "poll_me.txt")
        with open(path, "w") as f:
            f.write("data")
        time.sleep(1.0)
    finally:
        obs.stop()
        obs.join(timeout=5)

print(f"PollingObserver captured: {captured}")
assert obs.is_alive() is False